# Fine-tune the folder assistant (Qwen3.5-4B), all in one notebook

**Before you start:** Runtime > Change runtime type > **T4 GPU** > Save. Then run the cells from top to bottom
(the play button on each cell).

What happens:
1. **Step 1:** you upload `local-assistant.zip`.
2. **Part A, making the training data (about 3 to 5 hours):** a bigger model (Qwen3.5-9B) writes the ideal answer
   for 600 fake folder requests, 150 at a time. Answers that break a safety rule are thrown away automatically.
3. **Part B, fine-tuning (about 1 hour):** the 4B model learns from those answers. At the end your browser
   downloads the fine-tuned model as a `.gguf` file to run on your PC.

Colab deletes its files when it disconnects (this happens on the free plan). So after every 150 answers the
notebook downloads `local-assistant-progress.zip` to your computer. If Colab disconnects, run **Step 1** again
and upload the newest progress zip instead: it continues where it stopped.

Want to try Part B first? Run Step 1, skip Part A, and run Part B. Without data it does a 5-minute test on the
7 hand-written examples, which shows the notebook works.

In [ ]:
# Step 1: upload local-assistant.zip (or the newest local-assistant-progress.zip after a disconnect)
import os, sys, zipfile, shutil
from google.colab import files
PROJECT = "/content/local-assistant"
if not os.path.exists(f"{PROJECT}/assistant"):
    print("Choose the zip on your computer:")
    name = next(iter(files.upload()))
    zipfile.ZipFile(name).extractall("/content")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def save_progress():
    """Downloads the project with all data made so far. After a disconnect, upload it in Step 1."""
    shutil.make_archive("/content/local-assistant-progress", "zip", "/content", "local-assistant")
    files.download("/content/local-assistant-progress.zip")

print("Ready:", os.getcwd())

## Shortcut: reuse the answers from an earlier run

Already ran Part A before? Run the cell below instead of A1 to A3, then run **A4** and go on with **Part B**.
It takes the teacher's answers from your old `local-assistant-progress.zip` and A4 checks them again with
today's rules, so this saves 3 to 5 hours. If A4 says fewer than about 60 drafts pass, do Part A instead.

In [ ]:
# Shortcut: upload the newest local-assistant-progress.zip from your earlier run. Only its drafts are used;
# the code stays the new version from Step 1.
print("Choose your old local-assistant-progress.zip:")
name = next(iter(files.upload()))
with zipfile.ZipFile(name) as z:
    old = z.read("local-assistant/dataset/drafts.jsonl")
with open("dataset/drafts.jsonl", "wb") as f:
    f.write(old)
print(sum(1 for l in old.splitlines() if l.strip()), "old drafts copied. Now run A4 (skip A1 to A3), then Part B.")

## Part A: make the training data

In [ ]:
# A1: install Ollama and the teacher model (about 5 minutes)
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
TEACHER = "qwen3.5:9b"          # fits the free T4
!ollama pull {TEACHER}
!pip install -q ollama pypdf python-docx pillow send2trash

In [ ]:
# A2: make the fake folders and requests (once). 150 folders x 4 requests = 600 tasks.
SCENARIOS = 150                 # more = better model but longer; 300 gives 1,200 tasks
if not os.path.exists("dataset/tasks.jsonl"):
    !python dataset/make_scenarios.py --scenarios {SCENARIOS} --requests 4
!wc -l dataset/tasks.jsonl

In [ ]:
# A3: the teacher writes 150 answers (about 1 hour), then your browser downloads local-assistant-progress.zip.
# Run this cell again until it says "0 to go". After a disconnect: Step 1 (newest progress zip), A1, then this.
!python dataset/draft_answers.py --teacher {TEACHER} --limit 150
save_progress()

In [ ]:
# A4: use the drafts as training data and split them into train / validation / test.
# (Better quality: on your PC, unzip the progress zip, run `python dataset/review.py`, zip the folder
#  again, upload it in Step 1 and set AUTO_ACCEPT = False.)
import shutil, json
AUTO_ACCEPT = True
if AUTO_ACCEPT:
    sys.path.insert(0, "dataset")
    from validate import check
    drafts = [json.loads(l) for l in open("dataset/drafts.jsonl", encoding="utf-8") if l.strip()]
    kept = [d for d in drafts if not check(d)]   # check again: drafts from an older version missed newer rules
    with open("dataset/reviewed.jsonl", "w", encoding="utf-8") as f:
        f.writelines(json.dumps(d, ensure_ascii=False) + "\n" for d in kept)
    print(f"{len(kept)} of {len(drafts)} drafts pass today's checks")
!python dataset/make_questions.py
!python dataset/split.py
rejected = sum(1 for _ in open("dataset/rejected.jsonl")) if os.path.exists("dataset/rejected.jsonl") else 0
print(f"{rejected} drafts were rejected by the safety checks (see dataset/rejected.jsonl)")
save_progress()

In [ ]:
# A5: stop Ollama so the GPU is free for training
!pkill -f "ollama serve"; sleep 3; nvidia-smi --query-gpu=memory.used --format=csv

## Part B: fine-tune the 4B

If you start a new Colab session here, run **Step 1** first and upload the newest progress zip.

In [ ]:
%%capture
# B1: install Unsloth (copied from Unsloth's official Qwen3.5 notebook, about 5 minutes)
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# B2: load Qwen3.5-4B and add a LoRA adapter (only about 1% of the weights are trained)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less wasted GPU memory
from unsloth import FastVisionModel   # Qwen3.5 is a vision-language model; we train only its language part
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    # Unsloth advises 16-bit LoRA for Qwen3.5, but that needs more memory than the free T4's 15 GB, so the T4 uses 4-bit
    load_in_4bit = torch.cuda.get_device_properties(0).total_memory < 20 * 2**30,
    use_gradient_checkpointing = "unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 3407,
)
tok = getattr(processor, "tokenizer", processor)

In [ ]:
# B3: turn the data into training examples, with the model's own chat template ("no thinking", like the app)
from assistant.schema import TOOLS
from train import sft_data

TRAIN, VAL = "dataset/train.jsonl", "dataset/val.jsonl"
SMOKE_TEST = not os.path.exists(TRAIN)
if SMOKE_TEST:
    TRAIN = VAL = "dataset/seed_examples.jsonl"
    print("No training data yet: 5-step test run on the 7 hand-written examples.")

template = tok if getattr(tok, "chat_template", None) else processor
def apply_template(messages, add_generation_prompt, tools=TOOLS):  # questions have their own tools
    return template.apply_chat_template(messages, tools=tools, tokenize=False,
                                        add_generation_prompt=add_generation_prompt, enable_thinking=False)

MAX_LEN = 6144
train, skipped = sft_data.build(TRAIN, tok, apply_template, MAX_LEN)
val, _ = sft_data.build(VAL, tok, apply_template, MAX_LEN)
lengths = sorted(len(e["input_ids"]) for e in train)
print(f"train {len(train)} examples, val {len(val)}, skipped {skipped}")
print(f"tokens per example: median {lengths[len(lengths)//2]}, max {lengths[-1]}")
ex = train[1]
cut = next(i for i, l in enumerate(ex["labels"]) if l != -100)
print("...", tok.decode(ex["input_ids"][cut-60:cut]), "\n==== THE MODEL LEARNS THIS PART ====\n", tok.decode(ex["input_ids"][cut:]))

In [ ]:
# B4: train (about 1 hour with 600 tasks; 5 minutes for the test run)
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# On GPUs without bfloat16 (the T4) Unsloth trains Qwen3.5 in float32, because float16 gives NaN errors in its layers
forced32 = os.environ.get("UNSLOTH_FORCE_FLOAT32") == "1"
bf16 = torch.cuda.is_bf16_supported() and not forced32
FastVisionModel.for_training(model)
trainer = Trainer(
    model = model,
    args = TrainingArguments(
        output_dir = "/content/outputs",
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,      # long examples: 8 at once would run out of GPU memory
        gradient_accumulation_steps = 8,
        num_train_epochs = 2,
        max_steps = 5 if SMOKE_TEST else -1,
        learning_rate = 2e-4,
        warmup_steps = 0.05,                 # under 1 means a fraction: warm up for 5% of the steps
        lr_scheduler_type = "linear",
        weight_decay = 0.001,
        optim = "adamw_8bit",
        logging_steps = 1 if SMOKE_TEST else 10,
        eval_strategy = "no" if SMOKE_TEST else "epoch",
        prediction_loss_only = True,         # evaluation needs only the loss, not the logits for every token
        save_strategy = "no",
        bf16 = bf16, fp16 = not bf16 and not forced32,
        report_to = "none",
        seed = 3407,
        remove_unused_columns = False,
    ),
    train_dataset = Dataset.from_list(train),
    eval_dataset = Dataset.from_list(val),
    data_collator = DataCollatorForSeq2Seq(tok, padding = True, label_pad_token_id = -100),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} minutes, peak GPU memory {torch.cuda.max_memory_reserved() / 2**30:.1f} GB")

In [ ]:
# B5: quick look at what the trained model answers for 3 validation requests
import json
FastVisionModel.for_inference(model)
records = [json.loads(l) for l in open(VAL, encoding="utf-8") if l.strip()][:3]
for rec in records:
    prompt, target = sft_data.examples(rec)[-1]
    inputs = tok(apply_template(prompt, True, sft_data.tools_for(rec)), return_tensors="pt",
                 add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=800, do_sample=False)
    print("USER:", [m["content"] for m in prompt if m["role"] == "user"][-1])
    print("MODEL:", tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False), "\n" + "-" * 80)

In [ ]:
# B6: make the .gguf file for your PC (about 10 minutes)
import glob
NAME = "assistant-4b" + ("-test" if SMOKE_TEST else "")
model.save_pretrained(f"/content/{NAME}-lora")           # the small adapter, kept as a backup
processor.save_pretrained(f"/content/{NAME}-lora")
shutil.make_archive(f"/content/{NAME}-lora", "zip", "/content", f"{NAME}-lora")
model.save_pretrained_gguf("/content/gguf", processor, quantization_method = "q4_k_m")
!ls -lh /content/gguf*/

In [ ]:
# B7: download the results: the LoRA backup, then the .gguf (about 2.7 GB) in 300 MB parts, because Colab
# cancels single downloads over about 2 GB. Allow multiple downloads if asked, and keep this tab open until they finish.
import hashlib
files.download(f"/content/{NAME}-lora.zip")
src = next(p for p in glob.glob("/content/gguf*/*.gguf") if "q4_k_m" in p.lower() and "mmproj" not in p)
os.makedirs("/content/parts", exist_ok=True)
sha = hashlib.sha256()
with open(src, "rb") as f:
    n = 0
    while chunk := f.read(300 * 2**20):
        sha.update(chunk)
        n += 1
        with open(f"/content/parts/{os.path.basename(src)}.part{n:02d}", "wb") as out:
            out.write(chunk)
with open(f"/content/parts/{os.path.basename(src)}.sha256.txt", "w") as out:
    out.write(f"{sha.hexdigest()}  {os.path.basename(src)}  {os.path.getsize(src)} bytes  {n} parts\n")
print(open(f"/content/parts/{os.path.basename(src)}.sha256.txt").read())
for path in sorted(glob.glob("/content/parts/*")):
    files.download(path)
if os.path.exists("dataset/test.jsonl"):
    files.download("dataset/test.jsonl")   # held-out test requests, for the comparison on your PC

## Run the fine-tuned model on your PC

1. Move the downloaded `.partNN` files and the `.sha256.txt` file into `C:\Users\kerim\Projects\local-assistant\models\`,
   and `test.jsonl` into `dataset\`. Then, in `local-assistant`, join the parts into one `.gguf` file:
   `python train/join_parts.py models`
2. Start it with llama.cpp (already on your PC), with <file> replaced by the file's name:
   ```
   C:\Users\kerim\Projects\tools\llama.cpp\bin\llama-server.exe -m C:\Users\kerim\Projects\local-assistant\models\<file>.gguf --jinja -ngl 99 -c 8192 --port 8080
   ```
3. In another terminal in `local-assistant`:
   ```
   .venv\Scripts\activate
   set LLM_BACKEND=llama-server
   python cli.py --base sandbox
   python bench/evaluate.py dataset/test.jsonl
   ```
   Run the same evaluation with `set LLM_BACKEND=ollama` to compare the original 4B with the fine-tuned one.